In [10]:
import pandas as pd

# 1. 加载整个 2.xlsx 文件
xls = pd.ExcelFile('/kaggle/input/datasets/ericlin073233/program/2.xlsx')

# 2. 打印真实表名
print("2.xlsx 里的真实表名：", xls.sheet_names)

# 3. 按位置读取（绝不出错）
df_load = pd.read_excel(xls, sheet_name=1, header=0)  # 第 1 张表
df_pv = pd.read_excel(xls, sheet_name=0, header=0)    # 第 0 张表

# 4. 读取 1.xlsx
df_price = pd.read_excel('/kaggle/input/datasets/ericlin073233/program/1.xlsx', header=0)

# 5. 打印列名和行数
print("负载列名：", df_load.columns.tolist())
print("光伏列名：", df_pv.columns.tolist())
print("负载行数：", len(df_load))
print("光伏行数：", len(df_pv))

# 6. 直接提取数据
prices = df_price['电价'].values.astype(float)
loads = df_load.iloc[:, -1].values.astype(float) # 取最后一列数据
pvs = df_pv.iloc[:, -1].values.astype(float)    # 取最后一列数据

print(f"时间点数量: {len(prices)}")

2.xlsx 里的真实表名： ['小区负载', '光伏发电实际功率']
负载列名： ['日期\\时间', datetime.time(0, 10), datetime.time(0, 20), datetime.time(0, 30), datetime.time(0, 40), datetime.time(0, 50), datetime.time(1, 0), datetime.time(1, 10), datetime.time(1, 20), datetime.time(1, 30), datetime.time(1, 40), datetime.time(1, 50), datetime.time(2, 0), datetime.time(2, 10), datetime.time(2, 20), datetime.time(2, 30), datetime.time(2, 40), datetime.time(2, 50), datetime.time(3, 0), datetime.time(3, 10), datetime.time(3, 20), datetime.time(3, 30), datetime.time(3, 40), datetime.time(3, 50), datetime.time(4, 0), datetime.time(4, 10), datetime.time(4, 20), datetime.time(4, 30), datetime.time(4, 40), datetime.time(4, 50), datetime.time(5, 0), datetime.time(5, 10), datetime.time(5, 20), datetime.time(5, 30), datetime.time(5, 40), datetime.time(5, 50), datetime.time(6, 0), datetime.time(6, 10), datetime.time(6, 20), datetime.time(6, 30), datetime.time(6, 40), datetime.time(6, 50), datetime.time(7, 0), datetime.time(7, 10), datetime

In [31]:
import pandas as pd
import pulp

# 1. 读数据
df = pd.read_excel('/kaggle/input/datasets/ericlin073233/program/1.xlsx', header=0)
prices = df['电价'].values.astype(float)
loads = df['小区负载'].values.astype(float)
pvs = df['光伏发电预测功率'].values.astype(float)

T = 144

# 2. 建模型
prob = pulp.LpProblem("Microgrid_Opt", pulp.LpMinimize)

# 决策变量（所有单位统一为 kWh）
buy = pulp.LpVariable.dicts("buy", range(T), lowBound=0)
charge = pulp.LpVariable.dicts("charge", range(T), lowBound=0)
discharge = pulp.LpVariable.dicts("discharge", range(T), lowBound=0)
soc = pulp.LpVariable.dicts("soc", range(T+1), lowBound=1200, upBound=10800)

# 买电成本
prob += pulp.lpSum([prices[t] * buy[t] for t in range(T)])

for t in range(T):
    # 功率平衡
    # 买电(kWh) + 光伏(kW)*1/6 + 放电(kWh) = 负载(kW)*1/6 + 充电(kWh)
    prob += buy[t] + pvs[t] * (1/6) + discharge[t] == loads[t] * (1/6) + charge[t]
    
    # 储能变化（充放电量乘以效率）
    prob += soc[t+1] == soc[t] + 0.9 * charge[t] - (1/0.9) * discharge[t]
    
    # 物理充放电限制：每 10 分钟最多充放电 5000/6 kWh
    prob += charge[t] <= 5000 / 6
    prob += discharge[t] <= 5000 / 6

# 初始和结束储能
prob += soc[0] == 6000
prob += soc[T] >= 5900
prob += soc[T] <= 6100

prob.solve()
print("\n求解状态:", pulp.LpStatus[prob.status])

if pulp.LpStatus[prob.status] == 'Optimal':
    print(f"最优全天购电费用: {pulp.value(prob.objective):.2f} 元")
    for t in range(5):
        print(f"时间 {t+1}: 买电 {buy[t].varValue:.2f}, 充电 {charge[t].varValue:.2f}, 放电 {discharge[t].varValue:.2f}, 储能 {soc[t].varValue:.2f}")

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /usr/local/lib/python3.12/dist-packages/pulp/apis/../solverdir/cbc/linux/i64/cbc /tmp/1ed9fac3533440daa5cdf06dac013555-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /tmp/1ed9fac3533440daa5cdf06dac013555-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 584 COLUMNS
At line 2028 RHS
At line 2608 BOUNDS
At line 2899 ENDATA
Problem MODEL has 579 rows, 577 columns and 1299 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Presolve 278 (-301) rows, 565 (-12) columns and 986 (-313) elements
Perturbing problem by 0.001% of 1.3952 - largest nonzero change 1.1023881e-06 ( 0.00024987976%) - largest zero change 1.1017985e-06
0  Obj 0.12909306 Primal inf 76913.402 (177)
80  Obj 4156.1307 Primal inf 135393.99 (192)
160  Obj 18236.473 Primal inf 93973.657 (158)
240  Obj 20947.966 Primal inf 62348.354 (108)
320  Obj 336

In [32]:
import pandas as pd
import pulp

# 1. 读取数据
df = pd.read_excel('/kaggle/input/datasets/ericlin073233/program/1.xlsx', header=0)
prices = df['电价'].values.astype(float)
loads = df['小区负载'].values.astype(float)
pvs = df['光伏发电预测功率'].values.astype(float)
times = df['时间'].values

T = 144

# 2. 模型（和之前的一样，直接复用）
prob = pulp.LpProblem("Microgrid_Opt", pulp.LpMinimize)
buy = pulp.LpVariable.dicts("buy", range(T), lowBound=0)
charge = pulp.LpVariable.dicts("charge", range(T), lowBound=0)
discharge = pulp.LpVariable.dicts("discharge", range(T), lowBound=0)
soc = pulp.LpVariable.dicts("soc", range(T+1), lowBound=1200, upBound=10800)

prob += pulp.lpSum([prices[t] * buy[t] for t in range(T)])

for t in range(T):
    prob += buy[t] + pvs[t] * (1/6) + discharge[t] == loads[t] * (1/6) + charge[t]
    prob += soc[t+1] == soc[t] + 0.9 * charge[t] - (1/0.9) * discharge[t]
    prob += charge[t] <= 5000 / 6
    prob += discharge[t] <= 5000 / 6

prob += soc[0] == 6000
prob += soc[T] >= 5900
prob += soc[T] <= 6100

prob.solve()

if pulp.LpStatus[prob.status] == 'Optimal':
    total_cost = pulp.value(prob.objective)
    total_buy = sum([buy[t].varValue for t in range(T)])
    print(f"最优全天购电费用: {total_cost:.2f} 元")
    print(f"全天购电量: {total_buy:.2f} kWh")

    #  3. 生成题目要求的 Excel 表格 
    with pd.ExcelWriter('result1.xlsx') as writer:
        
        # 表1：购电量（按题目要求的特定 10 分钟段）
        # 找出 10:00-10:10, 12:00-12:10 等对应的索引
        target_times = ['10:00:00', '12:00:00', '14:00:00', '16:00:00', '18:00:00', '20:00:00']
        buy_data = []
        for i, t in enumerate(times):
            if t in target_times:
                # 注意：这里的索引 i 是 10分钟一个点的第 i 个，要对应上
                buy_data.append({
                    '时间段': f"{t}-{times[i+1]}",
                    '购电量(kWh)': buy[i].varValue
                })
        
        # 加上全天购电量和购电费
        buy_data.append({'时间段': '全天购电量', '购电量(kWh)': total_buy})
        buy_data.append({'时间段': '全天购电费', '购电量(kWh)': total_cost})
        
        df_table1 = pd.DataFrame(buy_data)
        df_table1.to_excel(writer, sheet_name='表1_购电量', index=False)
        
        # 表2：储能充放电量（每 4 小时汇总一次）
        # 每天 144 个 10 分钟点，4小时 = 24 个点
        charge_summary = []
        discharge_summary = []
        for i in range(6):  # 0-4, 4-8, 8-12, 12-16, 16-20, 20-24
            start_idx = i * 24
            end_idx = (i + 1) * 24
            charge_sum = sum([charge[t].varValue for t in range(start_idx, end_idx)])
            discharge_sum = sum([discharge[t].varValue for t in range(start_idx, end_idx)])
            charge_summary.append(charge_sum)
            discharge_summary.append(discharge_sum)
        
        # 构造表格2
        table2_data = {
            '时间段': ['0:00-4:00', '4:00-8:00', '8:00-12:00', '12:00-16:00', '16:00-20:00', '20:00-24:00'],
            '充电量(kWh)': charge_summary,
            '放电量(kWh)': discharge_summary
        }
        df_table2 = pd.DataFrame(table2_data)
        
        # 加上 0:00 和 24:00 的储电量
        df_table2.loc[len(df_table2)] = ['0:00 储电量', soc[0].varValue, '']
        df_table2.loc[len(df_table2)] = ['24:00 储电量', soc[T].varValue, '']
        
        df_table2.to_excel(writer, sheet_name='表2_充放电量', index=False)
        
        print("\n✅ 结果已成功保存到 result1.xlsx 文件中！")
        print("你可以点击左侧文件列表里的 result1.xlsx 下载查看。")

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /usr/local/lib/python3.12/dist-packages/pulp/apis/../solverdir/cbc/linux/i64/cbc /tmp/a119b5cfd9b0446b8989b60c3a05b472-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /tmp/a119b5cfd9b0446b8989b60c3a05b472-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 584 COLUMNS
At line 2028 RHS
At line 2608 BOUNDS
At line 2899 ENDATA
Problem MODEL has 579 rows, 577 columns and 1299 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Presolve 278 (-301) rows, 565 (-12) columns and 986 (-313) elements
Perturbing problem by 0.001% of 1.3952 - largest nonzero change 1.1023881e-06 ( 0.00024987976%) - largest zero change 1.1017985e-06
0  Obj 0.12909306 Primal inf 76913.402 (177)
80  Obj 4156.1307 Primal inf 135393.99 (192)
160  Obj 18236.473 Primal inf 93973.657 (158)
240  Obj 20947.966 Primal inf 62348.354 (108)
320  Obj 336